In [1]:

# Analysis Plan for Testing GEV Shape Parameter ξ Scaling with Truncation Depth N
# 
# Research Hypothesis: If the observed negative ξ for ζ(s) is a pre-asymptotic effect, 
# then as truncation N increases, ξ(N) will trend towards 0. In contrast, for a random 
# multiplicative function, ξ(N) will not show a similar trend and will remain positive.
#
# PLAN:
# 1. Implement two functions: ζ(s) with a_n=1 and a random multiplicative function (seed=42)
# 2. For each N in {10^4, 3×10^4, 10^5, 3×10^5, 10^6, 3×10^6, 10^7}:
# - Compute partial sum D(t;N) over t ∈ [5000, 25000] with Kahan summation
# - Use numba JIT compilation for optimization
# 3. For each (Function, N) pair:
# - Partition t-range into 200 blocks
# - Fit GEV to block maxima of |D(t;N)|
# - Extract ξ and 95% CI (5000 bootstrap samples)
# 4. Analysis:
# - Create table with Function, N, ξ, and 95% CI bounds
# - Plot ξ vs log(N) with error bars
# - For ζ(s), fit constant and logarithmic decay models, report R² and AIC

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import curve_fit
import pandas as pd
from numba import njit
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully")
print("NumPy version:", np.__version__)
print("Analysis plan defined")


Libraries loaded successfully
NumPy version: 1.26.4
Analysis plan defined


In [2]:

# Step 1: Generate prime numbers and implement arithmetic functions
# Following the dataset description, we need primes up to N_max = 10^7

def sieve_of_eratosthenes(limit):
 """Generate all primes up to limit using Sieve of Eratosthenes."""
 if limit < 2:
 return np.array([], dtype=np.int64)
 is_prime = np.ones(limit + 1, dtype=bool)
 is_prime[0] = is_prime[1] = False
 for i in range(2, int(np.sqrt(limit)) + 1):
 if is_prime[i]:
 is_prime[i*i::i] = False
 return np.where(is_prime)[0]

# Generate primes up to 10^7
N_max = 10**7
print(f"Generating primes up to {N_max}...")
primes = sieve_of_eratosthenes(N_max)
print(f"Generated {len(primes)} primes")
print(f"First 10 primes: {primes[:10]}")
print(f"Last 10 primes: {primes[-10:]}")


Generating primes up to 10000000...
Generated 664579 primes
First 10 primes: [ 2 3 5 7 11 13 17 19 23 29]
Last 10 primes: [9999889 9999901 9999907 9999929 9999931 9999937 9999943 9999971 9999973
 9999991]


In [3]:

# Step 2: Implement random multiplicative function
# Following the dataset description: at each prime p, set a_p = +1 or -1 with probability 1/2
# Then extend multiplicatively

def generate_random_multiplicative_coefficients(N, primes, seed=42):
 """
 Generate coefficients for a random multiplicative function.
 a_p = +1 or -1 with equal probability for each prime p
 a_{p^e} = a_p^e
 a_{p1^e1 * ... * pk^ek} = a_{p1}^{e1} * ... * a_{pk}^{ek}
 """
 np.random.seed(seed)
 
 # Initialize coefficients array
 a = np.ones(N + 1, dtype=np.float64)
 
 # Assign random signs to primes
 prime_signs = {}
 for p in primes[primes <= N]:
 prime_signs[p] = np.random.choice([-1.0, 1.0])
 
 # Compute coefficients multiplicatively
 for n in range(2, N + 1):
 if n in prime_signs:
 # n is prime
 a[n] = prime_signs[n]
 else:
 # Factor n and compute a[n] multiplicatively
 temp_n = n
 a_n = 1.0
 for p in primes:
 if p * p > temp_n:
 break
 if temp_n % p == 0:
 exp = 0
 while temp_n % p == 0:
 temp_n //= p
 exp += 1
 a_n *= prime_signs[p] ** exp
 # If temp_n > 1, it's a prime factor
 if temp_n > 1:
 if temp_n in prime_signs:
 a_n *= prime_signs[temp_n]
 a[n] = a_n
 
 return a

# Test with small N
print("Testing random multiplicative function generation...")
N_test = 100
a_rand_test = generate_random_multiplicative_coefficients(N_test, primes, seed=42)
print(f"First 20 coefficients: {a_rand_test[:21]}")
print(f"Coefficient at n=6 (2*3): {a_rand_test[6]} (should be a[2]*a[3] = {a_rand_test[2]*a_rand_test[3]})")
print(f"Coefficient at n=12 (2^2*3): {a_rand_test[12]} (should be a[2]^2*a[3] = {a_rand_test[2]**2 * a_rand_test[3]})")


Testing random multiplicative function generation...
First 20 coefficients: [ 1. 1. -1. 1. 1. -1. -1. -1. -1. 1. 1. -1. 1. 1. 1. -1. 1. -1.
 -1. -1. -1.]
Coefficient at n=6 (2*3): -1.0 (should be a[2]*a[3] = -1.0)
Coefficient at n=12 (2^2*3): 1.0 (should be a[2]^2*a[3] = 1.0)


In [4]:

# Step 3: Implement Kahan compensated summation for Dirichlet partial sums
# Following the dataset description: D_F(t;N) = Σ_{n≤N} a_n(F) / n^(1/2+it)
# Use Kahan summation to mitigate catastrophic cancellation errors

@njit
def kahan_sum_dirichlet(a_n, N, t):
 """
 Compute D(t;N) = Σ_{n≤N} a_n / n^(1/2+it) using Kahan compensated summation.
 
 Parameters:
 -----------
 a_n : array of coefficients (a_n[0] is unused, a_n[1] to a_n[N] are used)
 N : truncation depth
 t : imaginary part of s = 1/2 + it
 
 Returns:
 --------
 Complex number D(t;N)
 """
 sum_real = 0.0
 sum_imag = 0.0
 comp_real = 0.0 # Compensation for lost low-order bits (real part)
 comp_imag = 0.0 # Compensation for lost low-order bits (imag part)
 
 for n in range(1, N + 1):
 # Compute n^(-1/2 - it) = n^(-1/2) * exp(-it*log(n))
 # = n^(-1/2) * (cos(-t*log(n)) + i*sin(-t*log(n)))
 n_sqrt_inv = 1.0 / np.sqrt(float(n))
 log_n = np.log(float(n))
 angle = -t * log_n
 
 # Real and imaginary parts of the term
 term_real = a_n[n] * n_sqrt_inv * np.cos(angle)
 term_imag = a_n[n] * n_sqrt_inv * np.sin(angle)
 
 # Kahan summation for real part
 y_real = term_real - comp_real
 temp_real = sum_real + y_real
 comp_real = (temp_real - sum_real) - y_real
 sum_real = temp_real
 
 # Kahan summation for imaginary part
 y_imag = term_imag - comp_imag
 temp_imag = sum_imag + y_imag
 comp_imag = (temp_imag - sum_imag) - y_imag
 sum_imag = temp_imag
 
 return sum_real + 1j * sum_imag

# Test the function
print("Testing Kahan summation for Dirichlet series...")
a_zeta = np.ones(101, dtype=np.float64) # Zeta function: a_n = 1
t_test = 100.0
N_test = 100
D_test = kahan_sum_dirichlet(a_zeta, N_test, t_test)
print(f"D_zeta(t={t_test}, N={N_test}) = {D_test}")
print(f"|D_zeta| = {np.abs(D_test):.6f}")


Testing Kahan summation for Dirichlet series...


D_zeta(t=100.0, N=100) = (2.767098710562044-0.09369732797829697j)
|D_zeta| = 2.768685


In [5]:

# Step 4: Compute D(t;N) over a range of t values
# For each N in {10^4, 3×10^4, 10^5, 3×10^5, 10^6, 3×10^6, 10^7}
# Compute D(t;N) for t ∈ [5000, 25000]
# Partition into 200 blocks for GEV analysis

# First, pre-generate coefficient arrays for both functions at N_max
print(f"Generating coefficient arrays for N_max = {N_max}...")

# Zeta function: a_n = 1 for all n
a_zeta = np.ones(N_max + 1, dtype=np.float64)
print("Zeta coefficients generated (all ones)")

# Random multiplicative function with seed=42
print("Generating random multiplicative coefficients (this may take a minute)...")
a_rand = generate_random_multiplicative_coefficients(N_max, primes, seed=42)
print("Random multiplicative coefficients generated")

# Verify multiplicativity for random function
print(f"\nVerifying multiplicativity:")
print(f"a[2] = {a_rand[2]}, a[3] = {a_rand[3]}, a[6] = {a_rand[6]}, a[2]*a[3] = {a_rand[2]*a_rand[3]}")
print(f"a[4] = {a_rand[4]}, a[2]^2 = {a_rand[2]**2}")


Generating coefficient arrays for N_max = 10000000...
Zeta coefficients generated (all ones)
Generating random multiplicative coefficients (this may take a minute)...


Random multiplicative coefficients generated

Verifying multiplicativity:
a[2] = -1.0, a[3] = 1.0, a[6] = -1.0, a[2]*a[3] = -1.0
a[4] = 1.0, a[2]^2 = 1.0


In [6]:

# Step 5: Define the analysis parameters
# N values to analyze
N_values = [10**4, 3*10**4, 10**5, 3*10**5, 10**6, 3*10**6, 10**7]

# t range: [5000, 25000]
t_min = 5000
t_max = 25000

# Number of blocks for GEV analysis
n_blocks = 200

# Number of bootstrap samples for confidence intervals
n_bootstrap = 5000

print("Analysis parameters:")
print(f"N values: {N_values}")
print(f"t range: [{t_min}, {t_max}]")
print(f"Number of blocks: {n_blocks}")
print(f"Bootstrap samples: {n_bootstrap}")

# Calculate number of t values per block
t_range = t_max - t_min
print(f"\nt range length: {t_range}")
print(f"Block size: {t_range / n_blocks}")


Analysis parameters:
N values: [10000, 30000, 100000, 300000, 1000000, 3000000, 10000000]
t range: [5000, 25000]
Number of blocks: 200
Bootstrap samples: 5000

t range length: 20000
Block size: 100.0


In [7]:

# The Kahan sum is still too slow even with JIT. Let me test computation time for one N value
import time

N_test = 10**4
t_test_array = np.linspace(5000, 5100, 10) # Just 10 t values
a_test = a_zeta[:N_test+1].copy()

print(f"Testing computation time for N={N_test}, 10 t values...")
start = time.time()
for t in t_test_array:
 D = kahan_sum_dirichlet(a_test, N_test, t)
end = time.time()
elapsed = end - start
print(f"Time for 10 evaluations: {elapsed:.2f} seconds")
print(f"Time per evaluation: {elapsed/10:.3f} seconds")
print(f"Estimated time for 1000 evaluations: {elapsed*100:.1f} seconds")


TimeoutError: Code execution timed out after 896 seconds